# Understanding States and Nodes in LangGraph

This notebook provides a comprehensive guide to understanding how states and nodes work in LangGraph, and how they enable building sophisticated, stateful applications.

## What is LangGraph?

LangGraph is a library for building stateful, multi-actor applications with LLMs. It extends LangChain by adding:

- **Stateful Graphs**: Applications that maintain state across interactions
- **Multi-Actor Systems**: Multiple agents working together
- **Cyclic Behavior**: Loops and conditional routing
- **Visualization**: Easy-to-understand graph representations

## Key Concepts

### 1. States
A **State** is a data structure that holds information as it flows through your application. Think of it as a shared memory that all nodes can read from and write to.

### 2. Nodes
A **Node** is a function that processes the current state and returns updates to that state.

### 3. Edges
**Edges** define how the graph flows from one node to another.

### 4. Graphs
A **Graph** is the complete structure that defines how nodes are connected and how state flows through them.


In [1]:
# Import required libraries
from langgraph.graph import StateGraph, START, END
from typing_extensions import TypedDict
from typing import List, Optional
from langchain_core.documents import Document
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

print("✅ Libraries imported successfully!")


✅ Libraries imported successfully!


## 1. Understanding States

A State in LangGraph is defined using a TypedDict. This provides type safety and makes it clear what data your application will work with.

### Basic State Definition


In [2]:
# Example 1: Simple State for a RAG System
class RAGState(TypedDict):
    question: str
    context: List[Document]
    response: str
    confidence: Optional[float]

# Example 2: More Complex State for Multi-Agent System
class MultiAgentState(TypedDict):
    user_input: str
    current_agent: str
    conversation_history: List[str]
    retrieved_documents: List[Document]
    analysis_results: dict
    final_response: str
    next_action: Optional[str]

print("✅ State definitions created!")
print("RAGState fields:", list(RAGState.__annotations__.keys()))
print("MultiAgentState fields:", list(MultiAgentState.__annotations__.keys()))


✅ State definitions created!
RAGState fields: ['question', 'context', 'response', 'confidence']
MultiAgentState fields: ['user_input', 'current_agent', 'conversation_history', 'retrieved_documents', 'analysis_results', 'final_response', 'next_action']


## 2. Understanding Nodes

A Node is a function that:
1. Takes the current state as input
2. Performs some processing
3. Returns updates to the state

### Key Node Characteristics:
- **Stateless Functions**: Nodes don't maintain internal state
- **Pure Functions**: Same input should produce same output
- **State Updates**: Return a dictionary with the fields you want to update
- **Composable**: Can be easily combined and reused


In [3]:
# Initialize models for our examples
chat_model = ChatOllama(model="gpt-oss:20b", temperature=0.7)
embedding_model = OllamaEmbeddings(model="embeddinggemma:latest")

# Example 1: Simple Node that processes a question
def process_question(state: RAGState) -> RAGState:
    """Node that processes and validates the user's question."""
    question = state["question"]
    
    # Simple validation
    if len(question.strip()) < 3:
        return {"question": question, "response": "Please provide a more detailed question."}
    
    # Add some processing
    processed_question = f"Processed: {question}"
    
    return {"question": processed_question}

# Example 2: Node that retrieves context
def retrieve_context(state: RAGState) -> RAGState:
    """Node that retrieves relevant context for the question."""
    question = state["question"]
    
    # Simulate document retrieval (in real app, this would use a vector store)
    mock_documents = [
        Document(page_content="AI is transforming industries"),
        Document(page_content="Machine learning enables automation"),
        Document(page_content="Natural language processing helps computers understand text")
    ]
    
    # Simple keyword matching for demo
    relevant_docs = []
    for doc in mock_documents:
        if any(word in doc.page_content.lower() for word in question.lower().split()):
            relevant_docs.append(doc)
    
    return {"context": relevant_docs}

# Example 3: Node that generates a response
def generate_response(state: RAGState) -> RAGState:
    """Node that generates a response based on question and context."""
    question = state["question"]
    context = state["context"]
    
    # Create context string
    context_str = "\n".join([doc.page_content for doc in context])
    
    # Create prompt
    prompt = ChatPromptTemplate.from_template("""
    Context: {context}
    
    Question: {question}
    
    Answer the question based on the context. If the context doesn't contain enough information, say so.
    """)
    
    # Create chain
    chain = prompt | chat_model | StrOutputParser()
    
    # Generate response
    response = chain.invoke({"context": context_str, "question": question})
    
    return {"response": response}

print("✅ Node functions created!")


✅ Node functions created!


## 3. Building a Simple Graph

Now let's put it all together and build a simple graph with our nodes.


In [4]:
# Create a simple RAG graph
def create_simple_rag_graph():
    # Create the graph builder
    graph_builder = StateGraph(RAGState)
    
    # Add nodes to the graph
    graph_builder.add_node("process_question", process_question)
    graph_builder.add_node("retrieve_context", retrieve_context)
    graph_builder.add_node("generate_response", generate_response)
    
    # Define the flow
    graph_builder.add_edge(START, "process_question")
    graph_builder.add_edge("process_question", "retrieve_context")
    graph_builder.add_edge("retrieve_context", "generate_response")
    graph_builder.add_edge("generate_response", END)
    
    # Compile the graph
    return graph_builder.compile()

# Create and test the graph
rag_graph = create_simple_rag_graph()

print("✅ Simple RAG graph created!")
print("Graph structure:")
print(rag_graph)


✅ Simple RAG graph created!
Graph structure:


In [5]:
# Test the graph with a sample question
test_question = "What is machine learning?"

print(f"Testing with question: '{test_question}'")
print("=" * 50)

# Run the graph
result = rag_graph.invoke({"question": test_question})

print("Final state:")
print(f"Question: {result['question']}")
print(f"Context documents: {len(result['context'])}")
for i, doc in enumerate(result['context']):
    print(f"  {i+1}. {doc.page_content}")
print(f"Response: {result['response']}")


Testing with question: 'What is machine learning?'
Final state:
Question: Processed: What is machine learning?
Context documents: 2
  1. AI is transforming industries
  2. Machine learning enables automation
Response: Machine learning is a type of artificial intelligence that enables automation.
